# DLavie OS — LoRA Fine-Tuning

Notebook ini melatih model AI kustom menggunakan dataset dari DLavie OS.

Cukup klik **Run All** dan tunggu hasilnya.

In [ ]:
# Install library
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'peft', 'trl', 'accelerate', 'bitsandbytes', 'datasets'], check=True)
print('✅ Library siap')

In [ ]:
# Cek GPU
import torch
assert torch.cuda.is_available(), '❌ GPU tidak aktif! Aktifkan di Settings > Accelerator > GPU T4'
print('✅ GPU:', torch.cuda.get_device_name(0))
print('   VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
# Load dataset DLavie OS
import json, os

DATASET_PATH = '/kaggle/input/dlavie-training-dataset/dataset.jsonl'

samples = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            s = json.loads(line)
            if s.get('input') and s.get('output'):
                samples.append(s)

print(f'✅ Dataset dimuat: {len(samples)} samples')
print(f'\nContoh:')
print(f'  Q: {samples[0]["input"][:80]}')
print(f'  A: {samples[0]["output"][:80]}')

In [ ]:
# Konfigurasi training
BASE_MODEL  = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_NAME = 'dlavie-model-v1'
EPOCHS      = 3
LORA_RANK   = 16
BATCH_SIZE  = 4
MAX_SEQ_LEN = 512
print(f'✅ Config: {BASE_MODEL} | {EPOCHS} epochs | LoRA r={LORA_RANK} | {len(samples)} samples')

In [ ]:
# Format ke chat template
from datasets import Dataset

def fmt(s):
    return (f"<|system|>\nYou are DLavie OS, a powerful AI assistant.\n"
            f"<|user|>\n{s['input']}\n"
            f"<|assistant|>\n{s['output']}")

dataset = Dataset.from_dict({'text': [fmt(s) for s in samples]})
print(f'✅ Dataset diformat: {len(dataset)} baris')
print('\nContoh formatted:')
print(dataset[0]['text'][:200])

In [ ]:
# Load model dengan 4-bit quantization
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

print(f'Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading model (4-bit)...')
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True)

lora_cfg = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_RANK*2,
    target_modules=['q_proj','v_proj','k_proj','o_proj'],
    lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print('✅ Model siap!')

In [ ]:
# TRAINING
from trl import SFTTrainer, SFTConfig

print('🚀 Training dimulai...')
args = SFTConfig(
    output_dir=f'./{OUTPUT_NAME}',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy='epoch',
    fp16=True,
    max_seq_length=MAX_SEQ_LEN,
    report_to='none',
    dataset_text_field='text',
)

trainer = SFTTrainer(model=model, args=args, train_dataset=dataset, tokenizer=tokenizer)
trainer.train()
print('✅ Training selesai!')

In [ ]:
# Simpan model
save_path = f'./{OUTPUT_NAME}-final'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

import shutil
shutil.make_archive(OUTPUT_NAME, 'zip', save_path)

print(f'✅ Model disimpan: {save_path}')
print(f'✅ File ZIP: {OUTPUT_NAME}.zip')
print()
print('Untuk download: panel kiri → Output → /kaggle/working/ → klik file .zip → Download')

In [ ]:
# Test hasil training
from transformers import pipeline
pipe = pipeline('text-generation', model=model, tokenizer=tokenizer,
                max_new_tokens=150, temperature=0.7, do_sample=True)

prompt = '<|system|>\nYou are DLavie OS, a powerful AI assistant.\n<|user|>\nHello! Who are you?\n<|assistant|>\n'
out = pipe(prompt)[0]['generated_text']
print('Q: Hello! Who are you?')
print('A:', out[len(prompt):])